# Pré-Processamento do Dataset de Comentários

In [1]:
import pandas as pd
import re
import string
import nltk
import emoji
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

STOP_WORDS = set(stopwords.words('portuguese'))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Nathan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Nathan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Nathan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Limpeza do texto

A função a seguir cuida da seguinte parte de limpeza do texto:
* Converte emoji em texto (ex: "😍" em ":smiling_face_with_heart_eyes);
* Remove tags html;
* Converte todos os caracteres para minúsculo;
* Retira urls;
* Retira menções a outros usuários e hashtags;
* Remove pontuação;
* Retira stopwords;
* Padroniza risadas;
* Padroniza concordâncias/gritos;
* Remove espaços extras;
* Faz tokenização;

In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = emoji.demojize(text, language='pt')
    text = re.sub(r'(:[^:]+:)(?:\s*\1)+', r'\1', text)
    text = text.replace(':', ' ').replace('_', ' ')

    # Remove html
    text = re.sub(r'<.*?>', '', text)
    
    text = text.lower()

    # urls e menções
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#\w+', '', text)

    # Padroniza Risadas (kkkk, rsrs, haha, etc) para a tag "tagrisada"
    text = re.sub(r'\b(k{2,}|rs{2,}|a?ha{2,}|hu?a{2,}|as[uh]{2,})\b', 'tagrisada', text)
    
    # Padroniza Concordância/Gritos 
    # Transforma letras repetidas (3 ou mais) em apenas uma
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = re.sub(r'!{2,}', '!', text)
    text = re.sub(r'\?{2,}', '?', text)

    punctuation_to_remove = string.punctuation.replace('!', '').replace('?', '')
    text = text.translate(str.maketrans('', '', punctuation_to_remove))

    # Espaços extras
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)

    filtered = [w for w in tokens if w not in STOP_WORDS and len(w) > 2]

    return " ".join(filtered)

In [4]:
def preprocess_yt_comments(df):
    df_processed = df.copy()

    df_processed = df_processed.drop_duplicates(subset=['commentId'])

    df_processed['commentPublishedAt'] = pd.to_datetime(df_processed['commentPublishedAt'])
    df_processed['commentUpdatedAt'] = pd.to_datetime(df_processed['commentUpdatedAt'])

    df_processed['cleanedText'] = df_processed['commentText'].apply(clean_text)
    df_processed = df_processed[df_processed['cleanedText'] != ""]

    return df_processed

In [5]:
df = pd.read_csv('comments_shorts.csv')
df_final = preprocess_yt_comments(df)
df_final.head()
df_final.to_csv("comments_shorts_processed.csv", index=False)
